# 02 – Làm sạch dữ liệu IMDb

**Đầu vào:** `data/processed/imdb_raw_merged.csv` từ notebook 01.

**Đầu ra chính:** `data/processed/imdb_cleaned.csv`.

Quy tắc: loại dòng thiếu `title`, khử trùng theo IMDb ID hoặc (tên + năm), ép kiểu số, thay số phi lý thành giá trị thiếu; không tùy tiện xóa phim có điểm đánh giá thấp hoặc IQR outlier. **Không tự điền điểm IMDb còn thiếu** vì đây là nhãn dùng cho Regression.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'
input_path = DATA / 'imdb_raw_merged.csv'
if not input_path.exists():
    raise FileNotFoundError('Hãy chạy notebook 01_data_loading.ipynb trước!')

df = pd.read_csv(input_path, low_memory=False)
raw_rows = len(df)
print('Dữ liệu ban đầu:', df.shape)

Dữ liệu ban đầu: (4573, 13)


## 2.1. Chuẩn hóa cột và văn bản

In [2]:
text_cols = ['title', 'director', 'star_cast', 'certificates',
             'poster_url', 'genres', 'genre_1', 'genre_2', 'genre_3', 'imdb_id']
num_cols = ['imdb_rating', 'year', 'duration_minutes', 'metascore']
for col in text_cols + num_cols:
    if col not in df.columns:
        df[col] = pd.NA

for col in text_cols:
    s = df[col].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    df[col] = s.replace(['', 'nan', 'None', 'none', 'N/A', 'n/a', 'NULL', '-', '--', '\\N'], pd.NA)

print('Số Title thiếu:', df['title'].isna().sum())

Số Title thiếu: 0


## 2.2. Ép kiểu số, kiểm tra giá trị không hợp lệ

In [3]:
def parse_number(series):
    s = series.astype('string').str.replace(',', '', regex=False)
    return pd.to_numeric(s.str.extract(r'([-+]?\d+(?:\.\d+)?)', expand=False), errors='coerce')

def parse_year(series):
    s = series.astype('string')
    years = s.str.extract(r'\b((?:18|19|20|21)\d{2})\b', expand=False)
    return pd.to_numeric(years, errors='coerce').astype('Int64')

def parse_runtime(value):
    if pd.isna(value):
        return np.nan
    s = str(value).lower().strip()
    # Ví dụ: '2h 30m', '2 hr 30 min'
    hours = re.search(r'(\d+(?:\.\d+)?)\s*(?:hours?|hrs?|h)\b', s)
    minutes = re.search(r'(\d+(?:\.\d+)?)\s*(?:minutes?|mins?|min|m)\b', s)
    if hours:
        return float(hours.group(1)) * 60 + (float(minutes.group(1)) if minutes else 0)
    # Ví dụ: '142 min' hoặc 142
    numbers = re.search(r'\d+(?:\.\d+)?', s.replace(',', ''))
    return float(numbers.group(0)) if numbers else np.nan


df['imdb_rating'] = parse_number(df['imdb_rating'])
df['metascore'] = parse_number(df['metascore'])
df['duration_minutes'] = df['duration_minutes'].map(parse_runtime)
df['year'] = parse_year(df['year'])

from datetime import date
max_year = date.today().year + 5
invalid_counts = {}
for col, lower, upper in [
    ('imdb_rating', 0, 10),
    ('metascore', 0, 100),
    ('duration_minutes', 1, 10000),
    ('year', 1888, max_year),
]:
    invalid = df[col].notna() & ~df[col].between(lower, upper)
    invalid_counts[col] = int(invalid.sum())
    df.loc[invalid, col] = pd.NA

print('Số giá trị không hợp lệ được chuyển thành NaN:', invalid_counts)

Số giá trị không hợp lệ được chuyển thành NaN: {'imdb_rating': 0, 'metascore': 0, 'duration_minutes': 0, 'year': 0}


## 2.3. Gộp các cột thể loại, giữ nội dung có ý nghĩa

In [4]:
def merge_genres(row):
    values = []
    for col in ['genres', 'genre_1', 'genre_2', 'genre_3']:
        value = row[col]
        if pd.isna(value):
            continue
        for token in re.split(r'[,|;/]', str(value)):
            token = token.strip().title()
            if token and token.lower() not in {'unknown', 'nan', 'none', 'n/a'} and token not in values:
                values.append(token)
    return ', '.join(values) if values else pd.NA

df['genres'] = df.apply(merge_genres, axis=1)
df = df.drop(columns=['genre_1', 'genre_2', 'genre_3'])

# Lưu trạng thái thiếu trước khi điền 'Unknown' cho thuộc tính mô tả.
for col in ['imdb_rating', 'year', 'duration_minutes', 'metascore',
            'director', 'star_cast', 'certificates', 'genres']:
    df[f'{col}_missing'] = df[col].isna().astype('int8')

missing_before_fill = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isna().sum().values,
    'missing_percent': (df.isna().mean() * 100).round(2).values,
}).sort_values('missing_percent', ascending=False)

# Thuộc tính phân loại thiếu được đánh dấu rõ ràng.
for col in ['director', 'star_cast', 'certificates', 'genres']:
    df[col] = df[col].fillna('Unknown')

display(missing_before_fill.head(15))

,column,missing_count,missing_percent
11,imdb_id,4573,100.00
9,poster_url,400,8.75
0,title,0,0.00
1,imdb_rating,0,0.00
3,duration_minutes,0,0.00
2,year,0,0.00
4,metascore,0,0.00
5,certificates,0,0.00
7,star_cast,0,0.00
6,director,0,0.00


## 2.4. Xóa bản ghi thiếu tên phim và loại trùng

In [5]:
removed_missing_title = int(df['title'].isna().sum())
df = df.loc[df['title'].notna()].copy()

# Ưu tiên bản ghi có nhiều trường đã được điền hơn.
quality_fields = ['imdb_rating', 'year', 'duration_minutes', 'metascore', 'poster_url']
missing_flags = [c for c in df.columns if c.endswith('_missing')]
df['_quality'] = df[quality_fields].notna().sum(axis=1) - df[missing_flags].sum(axis=1)

df['_title_key'] = (df['title'].astype('string').str.casefold()
                    .str.replace(r'[^\w]+', ' ', regex=True).str.strip())
df['_year_key'] = df['year'].astype('string').fillna('unknown-year')
df = df.sort_values('_quality', ascending=False, kind='stable')

# Khi có IMDb ID, dùng ID; nếu không thì dùng tên + năm.
id_valid = df['imdb_id'].notna()
df['_dedup_key'] = np.where(
    id_valid,
    'ID|' + df['imdb_id'].astype('string').fillna(''),
    'TITLE|' + df['_title_key'].fillna('') + '|' + df['_year_key'],
)
removed_duplicates = int(df.duplicated('_dedup_key', keep='first').sum())
df = (df.drop_duplicates('_dedup_key', keep='first')
        .drop(columns=['_quality', '_title_key', '_year_key', '_dedup_key'])
        .reset_index(drop=True))

print('Dòng ban đầu:', raw_rows)
print('Xóa do thiếu tên phim:', removed_missing_title)
print('Xóa do trùng:', removed_duplicates)
print('Dòng sau làm sạch:', len(df))

Dòng ban đầu: 4573
Xóa do thiếu tên phim: 0
Xóa do trùng: 857
Dòng sau làm sạch: 3716


## 2.5. Kiểm tra ngoại lệ IQR

IQR dùng để *phát hiện*, không tự động xóa vì phim dài/ngắn bất thường có thể hoàn toàn hợp lệ.

In [6]:
def iqr_report(frame, columns):
    records = []
    for col in columns:
        values = pd.to_numeric(frame[col], errors='coerce').dropna()
        if len(values) == 0:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo = q1 - 1.5 * iqr
        hi = q3 + 1.5 * iqr
        records.append({
            'column': col, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
            'lower_bound': lo, 'upper_bound': hi,
            'outlier_count': int(((values < lo) | (values > hi)).sum()),
            'valid_count': len(values),
        })
    return pd.DataFrame(records)

outlier_table = iqr_report(df, ['imdb_rating', 'year', 'duration_minutes', 'metascore'])
display(outlier_table)

assert df['title'].notna().all()
assert df['imdb_rating'].dropna().between(0, 10).all()
assert df['metascore'].dropna().between(0, 100).all()
assert (df['duration_minutes'].dropna() > 0).all()
# Đã kiểm tra bằng bộ khóa _dedup_key trước khi xuất file.

,column,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,valid_count
0,imdb_rating,6.2,7.3,1.1,4.55,8.95,50,3716
1,year,1996.0,2017.0,21.0,1964.50,2048.50,142,3716
2,duration_minutes,98.0,117.0,19.0,69.50,145.50,237,3716
3,metascore,57.1,66.9,9.8,42.40,81.60,542,3716


## 2.6. Xuất file `imdb_cleaned.csv`

**Lưu ý:** Số thiếu ở các cột số (`metascore`, `duration_minutes`, `year`) vẫn để `NaN` để các thành viên mô hình sử dụng `SimpleImputer` **chỉ fit trên tập train**, tránh data leakage. Nhãn `imdb_rating` thiếu không được tự suy đoán.

In [7]:
clean_path = DATA / 'imdb_cleaned.csv'
df.to_csv(clean_path, index=False, encoding='utf-8-sig')
missing_before_fill.to_csv(DATA / 'missing_before_fill.csv', index=False, encoding='utf-8-sig')
outlier_table.to_csv(DATA / 'iqr_report.csv', index=False, encoding='utf-8-sig')
report = {
    'raw_rows': raw_rows,
    'removed_missing_title': removed_missing_title,
    'removed_duplicates': removed_duplicates,
    'clean_rows': len(df),
    'invalid_values_to_nan': invalid_counts,
    'notes': [
        'Không loại bỏ IQR outliers tự động.',
        'Không suy luận cơ chế MCAR/MAR/MNAR chỉ từ missing-count.',
        'Biến số thiếu được giữ lại để impute trên tập training.',
    ]
}
(DATA / 'cleaning_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', clean_path)
print('Kích thước:', df.shape)
display(df.head())

Đã lưu: e:\Nhập Môn AI\NhapMonPhanTichDuLieuVaAI-FinalProject\data\processed\imdb_cleaned.csv
Kích thước: (3716, 20)


,title,imdb_rating,year,duration_minutes,metascore,certificates,director,star_cast,source_file,poster_url,genres,imdb_id,imdb_rating_missing,year_missing,duration_minutes_missing,metascore_missing,director_missing,star_cast_missing,certificates_missing,genres_missing
0,End of the Spear,6.8,2005,108.0,45.0,PG-13,Jim Hanon,Louie LeonardoChad AllenJack Guzman,IMDb_Dataset_2.csv,https://m.media-amazon.com/images/M/MV5BMTYxOT...,Adventure,<NA>,0,0,0,0,0,0,0,0
1,Elvira Madigan,7.0,1967,91.0,66.0,PG,Bo Widerberg,Pia DegermarkThommy BerggrenLennart Malmer,IMDb_Dataset_2.csv,https://m.media-amazon.com/images/M/MV5BMmY2Nj...,Biography,<NA>,0,0,0,0,0,0,0,0
2,The Kid Stays in the Picture,7.3,2002,93.0,75.0,R,Nanette Burstein,Robert EvansEddie AlbertPeter Bart,IMDb_Dataset_2.csv,https://m.media-amazon.com/images/M/MV5BZjhiZm...,Documentary,<NA>,0,0,0,0,0,0,0,0
3,It Ain't Over,8.2,2022,99.0,79.0,PG,Sean Mullin,Andy AndresRoger AngellMarty Appel,IMDb_Dataset_2.csv,https://m.media-amazon.com/images/M/MV5BZWViYW...,Documentary,<NA>,0,0,0,0,0,0,0,0
4,Mahler,7.0,1974,115.0,66.0,PG,Ken Russell,Robert PowellGeorgina HaleLee Montague,IMDb_Dataset_2.csv,https://m.media-amazon.com/images/M/MV5BYzY4Mz...,Biography,<NA>,0,0,0,0,0,0,0,0
